# 🤖 Session 22: Programmable AI and AI Agents (Hands-On Lab)
> **Open Source Development Program**  
> Practical Workshop on Deterministic Structured Outputs, ReAct Loops, and Autonomous Tool-Using Agents.

---

### 🎯 What We Will Build Today:
1. **Module 1 & 2: Programmable AI Foundations**
   - Developer APIs vs Consumer Chat UIs
   - The Three Roles (`system`, `user`, `assistant`) and Determinism (`temperature=0.0`)
   - Zero-Shot vs Few-Shot Prompting
   - Overcoming "The Parsing Problem" with **Pydantic & Strict Structured Outputs**
2. **Module 3: 🧪 Lab 1 — The Expense Parser**
   - Transform chaotic, messy emails into clean, type-safe database records (`ExpenseReport`)
3. **Module 4 & 5: Rise of the AI Agents & The ReAct Pattern**
   - Breaking "The Wall" (Giving LLMs Hands)
   - The **ReAct Cycle**: `Reason (Thought)` $\rightarrow$ `Act (Tool Call)` $\rightarrow$ `Observe (Tool Output)` $\rightarrow$ `Finish`
   - Building custom Python Tools (`@tool`) with strict docstrings & function signatures
4. **Module 6: 🧪 Lab 2 — The Market Agent**
   - An autonomous financial agent that fetches live stock prices, performs safe math calculations, and searches market catalysts
5. **Module 7: 🤝 Multi-Agent Systems & Handoffs**
   - Specialization and Agent Collaboration: `Market Researcher Agent` $\rightarrow$ `Financial Analyst Agent`


---
## 📦 Step 0: Environment Setup & API Configuration
Let's install dependencies and securely load your API keys (`GOOGLE_API_KEY`, `OPENAI_API_KEY`, or `GROQ_API_KEY`).


In [ ]:
# Run this cell to install dependencies if not already installed
!pip install -q pydantic python-dotenv google-generativeai openai groq yfinance duckduckgo-search rich pandas tabulate streamlit


In [ ]:
from IPython.display import display
import os
import json
import ast
import operator
from typing import List, Literal, Optional, Dict, Any
from dotenv import load_dotenv
import pandas as pd
from pydantic import BaseModel, Field

# Load API Keys from .env
load_dotenv()
load_dotenv("../.env")

# Verify available provider
google_key = os.getenv("GOOGLE_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

provider = "mock"
gemini_model_name = "gemini-3.5-flash-lite"

if google_key:
    provider = "gemini"
    import google.generativeai as genai
    genai.configure(api_key=google_key)
    print(f"✅ Configured Google Gemini API (Model: {gemini_model_name})!")
elif openai_key:
    provider = "openai"
    import openai
    openai_client = openai.OpenAI(api_key=openai_key)
    print("✅ Configured OpenAI API (gpt-4o-mini)!")
elif groq_key:
    provider = "groq"
    from groq import Groq
    groq_client = Groq(api_key=groq_key)
    print("✅ Configured Groq API (llama-3.3-70b)!")
else:
    print("⚠️ No API Key found in .env! Running in Educational Simulation / Mock Mode.")
    print("💡 To use live LLMs, add GOOGLE_API_KEY, OPENAI_API_KEY, or GROQ_API_KEY to your .env file.")


---
## 📚 Module 1: The API Layer & Temperature (Slide 3–5)
### Why Developer APIs?
- **Chat UIs** are designed for humans with conversational filler words (*"Sure, I can help you with that!"*).
- **Developer APIs** enable machine-to-machine sync, deterministic outputs, and invisible background automation.

### The Three Roles:
1. `system`: Foundational operational rules & persona (e.g., *"You are a strict data extraction parser"*).
2. `user`: Raw incoming input string or prompt.
3. `assistant`: Model's previous responses for conversation memory.

### Temperature & Determinism:
- `temperature=0.0`: **Strict Code Mode**. Highly deterministic, predictable, and analytical.
- `temperature=0.7`: Balanced conversational chat.
- `temperature=1.0+`: High variability & creativity (high risk of hallucinations).


In [ ]:
def test_temperature_determinism(prompt: str, temp: float = 0.0, runs: int = 3):
    """Demonstrates how temperature affects output variance across multiple runs."""
    print(f"\n--- Testing Temperature: {temp} ({runs} runs) ---")
    for i in range(runs):
        if provider == "gemini":
            try:
                model = genai.GenerativeModel(gemini_model_name, generation_config={"temperature": temp})
                resp = model.generate_content(prompt)
                print(f"Run {i+1}: {resp.text.strip()}")
            except Exception as e:
                print(f"Run {i+1}: [Temp {temp}] AlgoTrader Execution Flow {i+1}")
        elif provider == "openai":
            resp = openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=temp
            )
            print(f"Run {i+1}: {resp.choices[0].message.content.strip()}")
        else:
            print(f"Run {i+1}: [Temp {temp}] Predictable JSON output #10{i+1}")

test_prompt = "Generate a single 3-word slogan for a high-frequency trading algorithm."
test_temperature_determinism(test_prompt, temp=0.0, runs=3)
test_temperature_determinism(test_prompt, temp=1.0, runs=3)


---
## 📚 Module 2: Prompting for Software & The Parsing Problem (Slide 6–10)
### The Parsing Problem (Slide 7)
If an LLM includes markdown backticks, greetings, or commentary (`"Sure, here is your JSON:"`), standard `json.loads()` will instantly raise a `JSONDecodeError` and crash production services!

Let's test this in action:


In [ ]:
# Sample messy text from a user email
messy_sample = "Bought a coffee for 4.50 at Starbucks yesterday, and spent $50 on gas at Shell."

def chatty_llm_call(text: str) -> str:
    prompt = f"Extract the expenses from this text as JSON:\n{text}"
    if provider == "gemini":
        try:
            model = genai.GenerativeModel(gemini_model_name, generation_config={"temperature": 0.7})
            return model.generate_content(prompt).text
        except Exception:
            return 'Sure! Here is the JSON output you requested:\n```json\n[{"vendor": "Starbucks", "amount": 4.50}]\n```\nHope this helps!'
    elif provider == "openai":
        resp = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )
        return resp.choices[0].message.content
    else:
        return 'Sure! Here is the JSON output you requested:\n```json\n[{"vendor": "Starbucks", "amount": 4.50}]\n```\nHope this helps!'

raw_chatty_output = chatty_llm_call(messy_sample)
print("Raw LLM Output:\n", raw_chatty_output)
print("-" * 50)

# Attempt to parse with json.loads
try:
    data = json.loads(raw_chatty_output)
    print("✅ json.loads() Succeeded!")
except Exception as err:
    print(f"❌ json.loads() FAILED with: {err}")
    print("💡 Notice how conversational text or markdown prevents direct programmatic consumption.")


### Defining the Blueprint with Pydantic (Slide 8 & 9)
Instead of regex and fragile string splitting, we define a **Pydantic Model**. The LLM is forced by API constraints to generate strictly valid JSON conforming to our schema.


In [ ]:
# Define strict Pydantic schemas for financial data
class ExpenseItem(BaseModel):
    vendor: str = Field(description="Name of the merchant, store, or service vendor")
    amount: float = Field(description="Monetary amount in USD")
    category: str = Field(description="Classified business expense category (e.g. Food & Beverage, Travel, Office, Software)")
    description: str = Field(description="Context or specific items purchased")

class ExpenseReport(BaseModel):
    expenses: List[ExpenseItem] = Field(description="List of all extracted expense items")
    total_amount: float = Field(description="Accurate sum total of all expenses")

# Inspect the auto-generated JSON Schema that the LLM receives
print("Generated JSON Schema for LLM:\n", json.dumps(ExpenseReport.model_json_schema(), indent=2))


---
## 🧪 Lab 1: The Expense Parser (Slide 11–13)
Let's build a production-grade parser function that converts messy emails, travel logs, and Slack receipts into structured Pydantic models with 100% type safety.


In [ ]:
def parse_expenses_structured(raw_text: str) -> ExpenseReport:
    """Extracts structured ExpenseReport using strict Schema enforcement."""
    system_instruction = (
        "You are an enterprise financial auditor. Extract all valid expense transactions from the text into "
        "the exact ExpenseReport schema. Accurately calculate total_amount and classify each item's category."
    )
    
    if provider == "gemini":
        try:
            model = genai.GenerativeModel(
                gemini_model_name,
                generation_config={
                    "temperature": 0.0,
                    "response_mime_type": "application/json",
                    "response_schema": ExpenseReport
                },
                system_instruction=system_instruction
            )
            resp = model.generate_content(raw_text)
            res_obj = ExpenseReport.model_validate_json(resp.text)
            if res_obj and res_obj.expenses:
                res_obj.total_amount = round(sum(item.amount for item in res_obj.expenses), 2)
            return res_obj
        except Exception as e:
            print("Gemini API fallback:", e)
    
    elif provider == "openai":
        try:
            completion = openai_client.beta.chat.completions.parse(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": raw_text}
                ],
                response_format=ExpenseReport,
                temperature=0.0
            )
            return completion.choices[0].message.parsed
        except Exception as e:
            print("OpenAI fallback:", e)
    
    # Fallback simulation
    return ExpenseReport(
        expenses=[
            ExpenseItem(vendor="Starbucks", amount=4.50, category="Food & Beverage", description="Coffee"),
            ExpenseItem(vendor="Shell", amount=50.00, category="Travel & Transportation", description="Gas refuel"),
            ExpenseItem(vendor="In-N-Out", amount=12.75, category="Food & Beverage", description="Burger"),
            ExpenseItem(vendor="CityGarage", amount=18.30, category="Travel & Transportation", description="Parking ticket")
        ],
        total_amount=85.55
    )


### Test Lab 1 with Complex Messy Inputs


In [ ]:
# Test Case 1: Travel Day Receipt Email
travel_email = """
Hey Team,
Submitting my travel expenses from yesterday's client visit:
Grabbed breakfast and a latte for $14.50 at Blue Bottle Coffee around 8 AM.
Took an Uber from SFO to Downtown for $48.20.
Had lunch with the client at Chipotle ($36.80 for both of us).
Later I took an afternoon Lyft back to the airport for $52.10 and bought a sandwich at Peet's Coffee for $11.25.
Please approve! - Alex
"""

report = parse_expenses_structured(travel_email)

# Convert to Pandas DataFrame for clean tabular presentation
df = pd.DataFrame([item.model_dump() for item in report.expenses])
print("=== EXTRACTED EXPENSE REPORT ===")
display(df)
print(f"💰 Calculated Total: ${report.total_amount:.2f}")


### ✍️ Student Exercise 1: Edge Cases & Negated Purchases
Test what happens when the user mentions an item they *did not* buy:
`"I bought a keyboard from Logitech for $89.99, but decided NOT to purchase the $450 4K monitor."`
Verify that your parser extracts only the verified purchase!


In [ ]:
tricky_email = """
Expense Log:
1. Team dinner at Olive Garden: $125.00
2. Bought Logitech MX Master 3 mouse from Best Buy for $99.99
3. Considered purchasing a new MacBook for $1,999.00 but decided to hold off and did not buy it.
4. Subway train tickets: $15.50
"""

tricky_report = parse_expenses_structured(tricky_email)
df_tricky = pd.DataFrame([item.model_dump() for item in tricky_report.expenses])
display(df_tricky)
print(f"Total: ${tricky_report.total_amount:.2f}")


---
## 📚 Module 4 & 5: Rise of AI Agents & The ReAct Loop (Slide 14–22)
### "The Wall" (Static LLM Limitations - Slide 14)
Standard LLMs are **"Brains in a Jar"**:
- ❌ Cannot access live stock prices or internet data.
- ❌ Cannot perform precise multi-step arithmetic reliably (hallucinates calculations).
- ❌ Cannot execute code, run SQL, or interact with local file systems.

### What is an AI Agent? (Slide 15)
$$\text{AI Agent} = \text{LLM (The Brain)} + \text{Tools (The Hands)} + \text{The Autonomous ReAct Loop}$$

### The ReAct Pattern (Slide 16–19):
```mermaid
graph TD
    UserQuery["User Request"] --> Thought1["🧠 Reason (Thought): Plan next step"]
    Thought1 --> Action1["⚡ Act: Call specific Tool with JSON args"]
    Action1 --> ToolExec["⚙️ Execute Tool in Python"]
    ToolExec --> Obs1["👁️ Observe: Read Tool Output"]
    Obs1 --> Thought2["🧠 Reason: Evaluate progress"]
    Thought2 --> FinalAns["🎯 Final Answer to User"]
```


---
## 🛠️ Defining Agent Tools ("The Hands" - Slide 20–22)
Every tool is a standard Python function with **type annotations** and **detailed docstrings**. The docstrings tell the LLM *when* and *how* to use the tool.


In [ ]:
# Simulated fallback cache for offline labs
STOCK_PRICE_CACHE = {
    "AAPL": 224.50,
    "MSFT": 428.15,
    "GOOGL": 182.30,
    "AMZN": 186.75,
    "NVDA": 121.40,
    "TSLA": 250.80,
    "META": 585.60,
}

def get_stock_price(ticker: str) -> float:
    """
    Fetches the real-time stock price for a given ticker symbol.
    Args:
        ticker: Uppercase stock ticker symbol (e.g. 'AAPL', 'MSFT', 'TSLA').
    Returns:
        float: The current stock price in USD.
    """
    clean_ticker = ticker.strip().upper().replace("$", "")
    try:
        import yfinance as yf
        stock = yf.Ticker(clean_ticker)
        if hasattr(stock, "fast_info") and stock.fast_info.last_price:
            return round(float(stock.fast_info.last_price), 2)
    except Exception:
        pass
    
    return STOCK_PRICE_CACHE.get(clean_ticker, 150.00)


def calculator(expression: str) -> float:
    """
    Safely calculates mathematical expressions (+, -, *, /, //, **, %).
    Args:
        expression: Math string (e.g., '15 * 428.15', '2500 / 121.40', '100 * (1 + 0.08)').
    Returns:
        float: The exact arithmetic result.
    """
    import re
    ops = {
        ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.FloorDiv: operator.floordiv,
        ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg, ast.UAdd: operator.pos
    }
    def _eval(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        elif isinstance(node, ast.BinOp):
            return ops[type(node.op)](_eval(node.left), _eval(node.right))
        elif isinstance(node, ast.UnaryOp):
            return ops[type(node.op)](_eval(node.operand))
        raise ValueError("Invalid math expression")
    
    clean_expr = str(expression).replace(",", "").replace("$", "").strip()
    clean_expr = re.sub(r"(\d+(?:\.\d+)?)\s*%", r"(\1 / 100)", clean_expr)
    return round(float(_eval(ast.parse(clean_expr, mode="eval").body)), 4)


def search_market_news(query: str) -> str:
    """
    Searches web news for recent market catalysts and company headlines.
    Args:
        query: Search query string (e.g., 'Microsoft cloud revenue growth').
    Returns:
        str: Relevant market news summaries.
    """
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=2))
            if results:
                return "\n".join([f"• {r['title']}: {r['body']}" for r in results])
    except Exception:
        pass
    return f"Market News for '{query}': Strong analyst ratings with steady quarterly earnings performance."

# Tool Registry
TOOL_REGISTRY = {
    "get_stock_price": get_stock_price,
    "calculator": calculator,
    "search_market_news": search_market_news,
}

print("✅ Registered Tools:", list(TOOL_REGISTRY.keys()))


---
## 🧪 Lab 2: The Market Agent (Slide 24–27)
Let's build the autonomous ReAct engine that drives our Market Agent.


In [ ]:
def parse_action_args(args_str: str):
    raw = args_str.strip()
    if not raw:
        return [], {}
    if raw.startswith("(") and raw.endswith(")"):
        raw = raw[1:-1].strip()
    if not raw:
        return [], {}
    try:
        call_expr = ast.parse(f"func({raw})").body[0].value
        pos_args = [ast.literal_eval(a) for a in call_expr.args]
        kw_args = {kw.arg: ast.literal_eval(kw.value) for kw in call_expr.keywords}
        if len(pos_args) == 1 and isinstance(pos_args[0], dict) and not kw_args:
            return [], pos_args[0]
        return pos_args, kw_args
    except Exception:
        pass
    try:
        val = json.loads(raw)
        if isinstance(val, dict):
            return [], val
        elif isinstance(val, list):
            return val, {}
        return [val], {}
    except Exception:
        pass
    try:
        val = ast.literal_eval(raw)
        if isinstance(val, dict):
            return [], val
        elif isinstance(val, (list, tuple)):
            return list(val), {}
        return [val], {}
    except Exception:
        pass
    return [raw], {}

AGENT_SYSTEM_PROMPT = """You are an autonomous Financial Intelligence Agent operating via the ReAct pattern.
You have access to the following tools:
1. get_stock_price(ticker="AAPL") -> float: Fetches the live stock price in USD.
2. calculator(expression="15 * 224.50") -> float: Evaluates exact arithmetic expressions. Never guess math!
3. search_market_news(query="TSLA deliveries") -> str: Fetches recent news catalysts.

You MUST follow this exact loop:
Thought: Reason about what you need to do.
Action: tool_name({"arg1": "val1"})
Observation: [The system will execute the tool and provide the answer here]
... (Repeat Thought/Action/Observation as needed)
Thought: I have gathered all information.
Final Answer: Complete, clearly explained final response with step-by-step numbers.

RULES:
- Emit exactly ONE Action at a time, then STOP and wait for the Observation.
- Always use the calculator tool for any multiplication, division, addition, or percentages.
- Always fetch actual stock prices with get_stock_price.
"""

class ReActMarketAgent:
    def __init__(self, max_steps: int = 6):
        self.max_steps = max_steps

    def _call_model(self, prompt: str) -> str:
        if provider == "gemini":
            try:
                from model_utils import call_gemini_with_fallback, extract_gemini_text
                resp = call_gemini_with_fallback(
                    prompt,
                    generation_config={"temperature": 0.0},
                    preferred_model=gemini_model_name
                )
                txt = extract_gemini_text(resp)
                if txt and txt.strip():
                    return txt.strip()
            except Exception as e:
                pass
        elif provider == "openai":
            try:
                resp = openai_client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0
                )
                return resp.choices[0].message.content
            except Exception:
                pass
        
        # Fallback mock agent response: dynamic ticker awareness
        detected_ticker = "MSFT"
        for t in ["MSFT", "AAPL", "NVDA", "TSLA", "AMZN", "GOOGL", "META"]:
            if t in prompt.upper():
                detected_ticker = t
                break

        price = STOCK_PRICE_CACHE.get(detected_ticker, 150.00)
        if "Observation:" not in prompt:
            return f'Thought: I need to look up the stock price of {detected_ticker}.\nAction: get_stock_price({{"ticker": "{detected_ticker}"}})'
        elif "calculator" not in prompt:
            return f'Thought: The {detected_ticker} stock price is ${price:.2f}. Now I will calculate 15 * {price}.\nAction: calculator({{"expression": "15 * {price}"}})'
        else:
            total_calc = round(15 * price, 2)
            return f'Thought: I have the calculated total.\nFinal Answer: {detected_ticker} is currently trading at ${price:.2f}. Purchasing 15 shares will cost exactly ${total_calc:,.2f}.'

    def run(self, query: str) -> str:
        history = f"{AGENT_SYSTEM_PROMPT}\n\nUser Query: {query}\n"
        print(f"\n{'='*65}\n🚀 [MARKET AGENT INITIALIZED] Query: '{query}'\n{'='*65}")

        for step in range(1, self.max_steps + 1):
            response = self._call_model(history)
            
            # Prioritize Action if present in response
            if "Action:" in response:
                action_idx = response.find("Action:")
                end_of_action = response.find("\n", action_idx)
                action_line = response[action_idx:].strip() if end_of_action == -1 else response[action_idx:end_of_action].strip()
                thought_part = response[:action_idx].strip()
                
                if thought_part:
                    print(f"\n🧠 [Step {step}] {thought_part}")
                print(f"⚡ [Tool Call] {action_line}")
                
                try:
                    raw_action = action_line.replace("Action:", "").strip()
                    tool_name = raw_action.split("(")[0].strip()
                    args_str = raw_action[len(tool_name):].strip()
                    pos_args, kw_args = parse_action_args(args_str)
                    
                    tool_fn = TOOL_REGISTRY.get(tool_name)
                    obs = tool_fn(*pos_args, **kw_args) if tool_fn else f"Tool {tool_name} not found"
                except Exception as e:
                    obs = f"Error: {e}"

                print(f"👁️ [Observation] {obs}")
                clean_step = (thought_part + "\n" + action_line).strip()
                history += f"{clean_step}\nObservation: {obs}\n"
                
            elif "Final Answer:" in response:
                thought = response.split("Final Answer:")[0].strip()
                final_ans = response.split("Final Answer:")[1].strip()
                if thought:
                    print(f"\n🧠 {thought}")
                print(f"\n🎯 [FINAL ANSWER]\n{final_ans}")
                return final_ans
            else:
                history += f"{response}\n"

        return "Max iterations reached."

# Instantiate and test Market Agent
market_agent = ReActMarketAgent()


### Running Complex Multi-Step Market Queries
Let's see our agent autonomously reason across multiple tool calls:


In [ ]:
# Query 1: Single stock math
market_agent.run("How much would it cost to buy 15 shares of Microsoft (MSFT)?")


In [ ]:
# Query 2: Multi-stock comparison & arithmetic
market_agent.run("What is the price of Apple (AAPL) and Tesla (TSLA)? If I buy 10 shares of each, what is the combined total cost?")


In [ ]:
# Query 3: Budget allocation
market_agent.run("If I have a budget of $3,500, how many whole shares of Nvidia (NVDA) can I buy?")


---
## 🤝 Module 7: Multi-Agent Systems & Handoffs (Slide 23)
### Specialization & Decomposition
In production architectures, rather than burdening a single agent with research, calculations, risk auditing, and creative report writing, we split responsibilities into specialized agents.

```mermaid
graph LR
    User --> Researcher["Agent 1: Market Researcher"]
    Researcher -->|Research Brief| Analyst["Agent 2: Financial Analyst & Writer"]
    Analyst --> Output["Executive Investment Memo"]
```


In [ ]:
class ResearchBrief(BaseModel):
    ticker: str
    price: float
    news_summary: str

class InvestmentMemo(BaseModel):
    title: str
    ticker: str
    allocated_capital: float
    shares: float
    executive_recommendation: str

def run_multi_agent_system(ticker: str, capital: float = 10000.0):
    print(f"\n{'='*65}\n👥 MULTI-AGENT WORKFLOW: {ticker.upper()} Allocation of ${capital:,.2f}\n{'='*65}")
    
    # --- AGENT 1: RESEARCHER ---
    print(f"🔍 [Agent 1: Researcher] Gathering live metrics and news for {ticker.upper()}...")
    price = get_stock_price(ticker)
    news = search_market_news(f"{ticker} quarterly earnings growth")
    brief = ResearchBrief(ticker=ticker.upper(), price=price, news_summary=news)
    print(f"   ↳ Research Brief compiled: Price=${price:.2f}")

    # --- HANDOFF TO AGENT 2: ANALYST ---
    print(f"\n📊 [Agent 2: Analyst & Writer] Ingesting Research Brief & calculating allocation...")
    shares = round(capital / brief.price, 2)
    
    memo = InvestmentMemo(
        title=f"Strategic Investment Brief: {brief.ticker}",
        ticker=brief.ticker,
        allocated_capital=capital,
        shares=shares,
        executive_recommendation=(
            f"Recommend allocating ${capital:,.2f} to acquire {shares} shares of {brief.ticker} "
            f"at current market price of ${brief.price:.2f}. "
            f"Growth Catalysts: {brief.news_summary[:150]}..."
        )
    )
    
    print(f"\n📋 [FINAL EXECUTIVE MEMO]\nTitle: {memo.title}")
    print(f"Target: {memo.ticker} | Capital: ${memo.allocated_capital:,.2f} ({memo.shares} shares)")
    print(f"Analysis: {memo.executive_recommendation}")

run_multi_agent_system("NVDA", 7500.0)


---
## 🏆 Lab Recap & Capstone Exercises

### 🔑 Key Takeaways:
1. **Developer APIs & Determinism**: Use `temperature=0.0` and structured schemas for software automation.
2. **The Parsing Problem**: Solved cleanly by enforcing **Pydantic models / JSON Schemas** at the API layer.
3. **The ReAct Loop**: Powers autonomous AI agents through iterative `Thought` $\rightarrow$ `Action` $\rightarrow$ `Observation`.
4. **Tools**: Python functions with descriptive docstrings serve as the LLM's hands.
5. **Multi-Agent Systems**: Specialization and clean handoffs allow scaling complex workflows.

---

### 🚀 Student Capstone Challenges:
1. **Challenge A: Currency Normalization Tool**
   - Add a tool `convert_currency(amount: float, from_curr: str, to_curr: str) -> float` to the Market Agent and query prices in EUR/GBP.
2. **Challenge B: Expense Category Tax Estimator**
   - Extend Lab 1's `ExpenseReport` schema to automatically calculate local VAT/GST per category!
3. **Challenge C: SQL / Database Agent**
   - Connect an agent to an in-memory SQLite database using Python's `sqlite3` module.
